# NEXUS Phase 4 — PPO Training: Unitree Go2 Navigation

**Hardware:** Kaggle T4 GPU (16GB)  
**Training:** 500,000 steps PPO  
**Target:** >85% success rate  
**Output:** `models/ppo_go2/policy.zip`

> Run this notebook on Kaggle with GPU accelerator enabled.

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install stable-baselines3[extra] gymnasium torch tensorboard

## Cell 2 — Go2Env (standalone, no ROS2)

In [ ]:
# go2_env.py embedded here — Kaggle has no ROS2
# Exact copy of backend/modules/robo_rl/go2_env.py

import gymnasium as gym
import numpy as np
from gymnasium import spaces
from typing import Optional, Tuple, Dict


class Go2Env(gym.Env):
    metadata = {'render_modes': ['human', 'rgb_array']}

    ARENA_LIMIT = 3.0
    GOAL_RADIUS  = 0.3

    REWARD_GOAL_REACHED  =  100.0
    REWARD_COLLISION     = -100.0
    REWARD_MOVING_TOWARD =    0.1
    REWARD_TIMESTEP      =   -0.01
    REWARD_BOUNDARY      =   -1.0

    MAX_STEPS = 1000

    ZONES = {
        'A': np.array([-2.5,  2.5, 0.0]),
        'B': np.array([ 2.5,  2.5, 0.0]),
        'C': np.array([-2.5, -2.5, 0.0]),
        'D': np.array([ 2.5, -2.5, 0.0]),
    }

    def __init__(self, render_mode=None, target_zone=None):
        super().__init__()
        self.render_mode = render_mode
        self.target_zone = target_zone

        obs_low = np.array(
            [-3.,-3.,0.]+[-1.,-1.,-1.,-1.]+[-1.,-1.,-1.]+[0.0]*12+[-0.5,-1.0],
            dtype=np.float32)
        obs_high = np.array(
            [3.,3.,1.]+[1.,1.,1.,1.]+[1.,1.,1.]+[5.0]*12+[0.5,1.0],
            dtype=np.float32)
        self.observation_space = spaces.Box(obs_low, obs_high, dtype=np.float32)
        self.action_space = spaces.Box(
            low=np.array([0.0,-1.0],dtype=np.float32),
            high=np.array([0.5, 1.0],dtype=np.float32), dtype=np.float32)

        self._robot_pos  = np.zeros(3, dtype=np.float32)
        self._robot_quat = np.array([0.,0.,0.,1.], dtype=np.float32)
        self._robot_vel  = np.zeros(2, dtype=np.float32)
        self._goal_pos   = np.zeros(3, dtype=np.float32)
        self._lidar      = np.ones(12, dtype=np.float32)*5.0
        self._step_count = 0
        self._prev_dist  = float('inf')
        self._obstacles  = []

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._robot_pos = np.array([
            self.np_random.uniform(-2.5, 2.5),
            self.np_random.uniform(-2.5, 2.5), 0.0], dtype=np.float32)
        yaw = self.np_random.uniform(-np.pi, np.pi)
        self._robot_quat = self._yaw_to_quat(yaw)
        self._robot_vel  = np.zeros(2, dtype=np.float32)
        if self.target_zone and self.target_zone in self.ZONES:
            self._goal_pos = self.ZONES[self.target_zone].copy()
        else:
            zone = self.np_random.choice(list(self.ZONES.keys()))
            self._goal_pos = self.ZONES[zone].copy()
        while np.linalg.norm(self._robot_pos[:2]-self._goal_pos[:2]) < 1.0:
            self._robot_pos[:2] = self.np_random.uniform(-2.5,2.5,2)
        self._obstacles  = self._generate_obstacles(n=self.np_random.integers(3,7))
        self._step_count = 0
        self._prev_dist  = self._dist_to_goal()
        self._lidar      = self._compute_lidar()
        return self._get_obs(), {'goal_pos': self._goal_pos.copy()}

    def step(self, action):
        action    = np.clip(action, self.action_space.low, self.action_space.high)
        linear_x  = float(action[0])
        angular_z = float(action[1])
        dt = 0.1
        yaw = self._quat_to_yaw(self._robot_quat)
        yaw += angular_z*dt
        self._robot_pos[0] += linear_x*np.cos(yaw)*dt
        self._robot_pos[1] += linear_x*np.sin(yaw)*dt
        self._robot_quat    = self._yaw_to_quat(yaw)
        self._robot_vel     = np.array([linear_x,angular_z],dtype=np.float32)
        self._lidar         = self._compute_lidar()
        reward, terminated  = self._compute_reward()
        self._step_count   += 1
        truncated           = self._step_count >= self.MAX_STEPS
        self._prev_dist     = self._dist_to_goal()
        return self._get_obs(), reward, terminated, truncated, {'dist': self._dist_to_goal()}

    def _compute_reward(self):
        dist = self._dist_to_goal()
        if dist < self.GOAL_RADIUS:                          return self.REWARD_GOAL_REACHED, True
        if self._check_collision():                          return self.REWARD_COLLISION,    True
        r = 0.0
        if self._check_boundary():                           r += self.REWARD_BOUNDARY
        if (self._prev_dist - dist) > 0:                     r += self.REWARD_MOVING_TOWARD
        r += self.REWARD_TIMESTEP
        return r, False

    def _get_obs(self):
        gv = self._goal_pos - self._robot_pos
        gd = (gv / (np.linalg.norm(gv)+1e-8)).astype(np.float32)
        return np.concatenate([self._robot_pos,self._robot_quat,gd,self._lidar,self._robot_vel]).astype(np.float32)

    def _dist_to_goal(self):
        return float(np.linalg.norm(self._robot_pos[:2]-self._goal_pos[:2]))

    def _check_collision(self):
        for (ox,oy,r) in self._obstacles:
            if np.linalg.norm(self._robot_pos[:2]-np.array([ox,oy])) < (r+0.25): return True
        return False

    def _check_boundary(self):
        return abs(self._robot_pos[0])>self.ARENA_LIMIT or abs(self._robot_pos[1])>self.ARENA_LIMIT

    def _compute_lidar(self):
        yaw=self._quat_to_yaw(self._robot_quat)
        angles=yaw+np.linspace(0,2*np.pi,12,endpoint=False)
        dists=np.full(12,5.0,dtype=np.float32)
        for i,angle in enumerate(angles):
            dx,dy=np.cos(angle),np.sin(angle)
            for t in np.arange(0.1,5.0,0.05):
                px=self._robot_pos[0]+dx*t; py=self._robot_pos[1]+dy*t
                hit=False
                if abs(px)>=self.ARENA_LIMIT or abs(py)>=self.ARENA_LIMIT: dists[i]=t; hit=True
                else:
                    for (ox,oy,r) in self._obstacles:
                        if np.linalg.norm(np.array([px-ox,py-oy]))<r: dists[i]=t; hit=True; break
                if hit: break
        return np.clip(dists,0.0,5.0)

    def _generate_obstacles(self,n):
        obs=[]
        for _ in range(n):
            for _ in range(50):
                ox=self.np_random.uniform(-2.5,2.5); oy=self.np_random.uniform(-2.5,2.5); r=0.2
                if (np.linalg.norm([ox-self._robot_pos[0],oy-self._robot_pos[1]])>0.8 and
                    np.linalg.norm([ox-self._goal_pos[0], oy-self._goal_pos[1]])>0.8):
                    obs.append((ox,oy,r)); break
        return obs

    @staticmethod
    def _yaw_to_quat(yaw):
        return np.array([0.,0.,np.sin(yaw/2),np.cos(yaw/2)],dtype=np.float32)

    @staticmethod
    def _quat_to_yaw(q):
        return 2.0*np.arctan2(float(q[2]),float(q[3]))

    def render(self): pass
    def close(self): pass


# Quick sanity check
env = Go2Env()
obs, _ = env.reset(seed=0)
assert obs.shape == (24,)
print(f'✅ Go2Env OK — obs shape: {obs.shape}, dtype: {obs.dtype}')

## Cell 3 — GPU check

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {device}')

## Cell 4 — Vectorized environment (8 parallel)

In [ ]:
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

N_ENVS = 8   # 8 parallel envs on T4

vec_env = make_vec_env(
    Go2Env,
    n_envs=N_ENVS,
    vec_env_cls=SubprocVecEnv,
    seed=42,
)

print(f'✅ VecEnv ready — {N_ENVS} parallel envs')
print(f'   obs  space : {vec_env.observation_space}')
print(f'   action space: {vec_env.action_space}')

## Cell 5 — PPO model setup

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, CallbackList
)
import os

os.makedirs('models/ppo_go2', exist_ok=True)
os.makedirs('logs/ppo_go2',   exist_ok=True)

model = PPO(
    policy        = 'MlpPolicy',
    env           = vec_env,
    learning_rate = 3e-4,
    n_steps       = 2048,
    batch_size    = 64,
    n_epochs      = 10,
    gamma         = 0.99,
    gae_lambda    = 0.95,
    clip_range    = 0.2,
    ent_coef      = 0.01,
    verbose       = 1,
    tensorboard_log = 'logs/ppo_go2',
    device        = device,
)

total_params = sum(p.numel() for p in model.policy.parameters())
print(f'✅ PPO model ready')
print(f'   Policy params : {total_params:,}')
print(f'   Device        : {device}')

## Cell 6 — Callbacks (checkpoint + eval)

In [ ]:
# Eval env — separate from training, single env
eval_env = make_vec_env(Go2Env, n_envs=4, seed=99)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path = 'models/ppo_go2/',
    log_path             = 'logs/ppo_go2/eval/',
    eval_freq            = 25_000,     # evaluate every 25k steps
    n_eval_episodes      = 20,
    deterministic        = True,
    verbose              = 1,
)

checkpoint_callback = CheckpointCallback(
    save_freq  = 100_000,              # checkpoint every 100k steps
    save_path  = 'models/ppo_go2/checkpoints/',
    name_prefix = 'ppo_go2',
    verbose    = 1,
)

callbacks = CallbackList([eval_callback, checkpoint_callback])
print('✅ Callbacks ready')
print('   Eval  every : 25,000 steps (20 episodes)')
print('   Checkpoint  : every 100,000 steps')

## Cell 7 — TRAIN (500,000 steps, ~3-4 hours on T4)

In [ ]:
import time

TOTAL_TIMESTEPS = 500_000

print(f'🚀 Starting PPO training')
print(f'   Total steps : {TOTAL_TIMESTEPS:,}')
print(f'   Parallel envs: {N_ENVS}')
print(f'   Effective steps/update: {N_ENVS * 2048:,}')
print()

t0 = time.time()

model.learn(
    total_timesteps = TOTAL_TIMESTEPS,
    callback        = callbacks,
    progress_bar    = True,
    tb_log_name     = 'ppo_go2_run1',
)

elapsed = time.time() - t0
print(f'\n✅ Training complete in {elapsed/3600:.2f} hours')

# Save final policy
model.save('models/ppo_go2/policy')
print('✅ Saved: models/ppo_go2/policy.zip')

## Cell 8 — Evaluate: success rate

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

# Load best model (saved by EvalCallback)
best_model = PPO.load('models/ppo_go2/best_model', env=eval_env)

mean_reward, std_reward = evaluate_policy(
    best_model, eval_env,
    n_eval_episodes=100,
    deterministic=True,
)

print(f'Mean reward : {mean_reward:.2f} ± {std_reward:.2f}')

# Manual success rate (goal reached = reward > 50)
successes = 0
N_EVAL    = 100

for ep in range(N_EVAL):
    obs = eval_env.reset()
    done = [False]*4
    ep_reward = [0.0]*4
    while not all(done):
        action, _ = best_model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = eval_env.step(action)
        for i, (r, d) in enumerate(zip(rewards, dones)):
            if not done[i]:
                ep_reward[i] += r
                if d:
                    done[i] = True
                    if ep_reward[i] > 50:
                        successes += 1
    if ep % 10 == 0:
        print(f'Episode {ep*4}/{N_EVAL*4} — running success: {successes}/{ep*4+4}')

success_rate = successes / (N_EVAL * 4) * 100
print(f'\n🎯 Success rate : {success_rate:.1f}%')
print(f'   Target       : 85.0%')
print(f'   Status       : {"✅ PASS" if success_rate >= 85 else "❌ FAIL — need more training"}')

## Cell 9 — Reward curve plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Load eval log
eval_log = np.load('logs/ppo_go2/eval/evaluations.npz')

timesteps    = eval_log['timesteps']
mean_rewards = eval_log['results'].mean(axis=1)
std_rewards  = eval_log['results'].std(axis=1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(timesteps, mean_rewards, color='#2196F3', linewidth=2, label='Mean reward')
ax.fill_between(timesteps,
                mean_rewards - std_rewards,
                mean_rewards + std_rewards,
                alpha=0.2, color='#2196F3')
ax.axhline(y=85, color='#4CAF50', linestyle='--', linewidth=1.5, label='Target (85)')
ax.set_xlabel('Training steps', fontsize=12)
ax.set_ylabel('Mean reward', fontsize=12)
ax.set_title('NEXUS Phase 4 — PPO Go2 Navigation Training Curve', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('models/ppo_go2/reward_curve.png', dpi=150)
plt.show()
print('✅ Saved: models/ppo_go2/reward_curve.png')

## Cell 10 — Download policy.zip

In [ ]:
# Download from Kaggle output
import shutil, os

# Copy to /kaggle/working for download
shutil.copy('models/ppo_go2/policy.zip',       '/kaggle/working/policy.zip')
shutil.copy('models/ppo_go2/best_model.zip',   '/kaggle/working/best_model.zip')
shutil.copy('models/ppo_go2/reward_curve.png', '/kaggle/working/reward_curve.png')

print('✅ Files ready for download in Kaggle output tab:')
for f in ['policy.zip','best_model.zip','reward_curve.png']:
    size = os.path.getsize(f'/kaggle/working/{f}')
    print(f'   {f} — {size/1024:.1f} KB')

print()
print('After download, put these files in your local project:')
print('   policy.zip      → models/ppo_go2/policy.zip')
print('   best_model.zip  → models/ppo_go2/best_model.zip')
print('   reward_curve.png → models/ppo_go2/reward_curve.png')